<center>
<div>
<img src="chroma_logo.webp" width="270"/>
</div>
</center>


**Resum.** Aquest és un notebook introductori de ChromaDB. ChromaDB és una base de dades vectorial que permet construir fàcilment aplicacions d'intel·ligència artificial amb *embeddings* o vectors. En aquest notebook il·lustrarem com fer servir ChromaDB mitjançant un cas d'ús: disseny d'un sistema de preguntes i respostes (QA).

## Vista general de l'aplicació

Un sistema QA rep com a entrada una pregunta i dona com a resultat una resposta que respon la pregunta. El sistema que implementarem té la següent arquitectura:


<center>
<div>
<img src="overview.png"/>
</div>
</center>

L'usuari introdueix una pregunta al sistema, la qual és introduïda com entrada al retriever (implementat usant ChromaDB) que accedeix a la base de coneixement per recuperar contextos que potencialment contenen la informació necessària per respondre la pregunta. Després aquests contextos s'introdueixen en un generador de respostes (en el nostre cas, farem servir un model d'OpenAI) que genera la resposta a la pregunta.


## Construcció de la BBDD amb ChromaDB (retriever)

Com a base de coneixement al nostre sistema farem servir el dataset [SQUAD](https://paperswithcode.com/dataset/squad). Es tracta d'un dataset que conté preguntes, les seves respostes, i contextos per respondre les preguntes. En el següent tros de codi carregarem el dataset i obtindrem una llista de parells (títols, contextos). Els títols indiquen el tema del context.

In [ ]:
from datasets import load_dataset

squad = load_dataset("squad")["train"]
contexts = squad["context"]
titles = squad["title"]
items = list(set(zip(titles, contexts)))
items[10]

Un cop carregat el dataset, construirem la base de dades vectorial amb ChromaDB i, per tant, el retriever. Així doncs, generem un client persistent que desarà la base de dades en el directori `./database`.

In [ ]:
import chromadb

client = chromadb.PersistentClient(path="./database")

A ChromaDB, una col·lecció és on emmagatzemarem els nostres vectors o embeddings, documents i metadades addicionals. Així doncs, creem una col·lecció anomenada `squad`.

In [ ]:
collection = client.get_or_create_collection(name="squad")

En aquesta col·lecció, emmagatzemarem els parells (títol, context):
- El títol es desarà com a metadada.
- El context és el que es transformarà en vector, per tal de ser indexat.

Internament, cadascun dels contextos es transforma en vectors fent servir el model transfomer [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) (model usat por ChromaDB per defecte). Aquests vectors són els que, finalment, s'indexen. D'aquesta manera, quan arriba una pregunta per part d'un usuari, aquesta pregunta es transforma en vector (usant el mateix model) i es compara de manera eficient amb cadascun dels contextos per identificar els top $k$ millors contextos que puguin respondre la pregunta.


A tall d'exemple, anem a afegir a la col·lecció únicament un element.

In [ ]:
id = "-1"
document = items[0][1]
title = items[0][0]
print(f"Document a afegir:\n{document}")
print(f"Títol: {title}")

Per afegir elements a la col·lecció fent servir el mètode `add`. Aquest mètode accepta una llista de documents, metadades i identificadors.

In [ ]:
collection.add(
    documents=[document],
    metadatas=[{"title": title}],
    ids=[id]
)

Podem fer servir el mètode `get` per visualitzar el que hem inserit. Si incloem els embeddings a la crida podem observar que ja ha estat assignat un vector al document.

In [ ]:
collection.get("-1", include=["embeddings", "documents", "metadatas"])

Per borrar un document podem fer servir el mètode `delete` amb el seu identificador.

In [ ]:
collection.delete("-1")

Seguint amb la construcció del sistema, al següent codi, recorrem els parells i els afegim a la base de dades. Tingueu paciència, la seva execució ha de trigar uns minuts.

In [ ]:
documents = []
metadatas = []
ids = []
for j, item in enumerate(items):
    ids.append(str(j))
    documents.append(item[1])
    metadatas.append({"title": item[0]})

collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

Un cop poblada la base de dades podem realitzar-hi consultes. Per exemple, podem preguntar Què és un SGBD? restringint el nombre de resultats a 1.

In [ ]:
results = collection.query(
    query_texts=["What is a DBMS?"],
    n_results=1,
    include=["documents"]
)
results

El mètode `query` ens permet indicar què volem tornar apart del document. Per exemple, al següent tros de codi executem la mateixa consulta però obtenim el document, les metadades i les distàncies del resultat amb la pregunta realitzada. A l'hora de fer la consulta, el text de la consulta (`"What is a DBMS?"`) és transformat a un vector i és compara amb cadascun dels vectors de la base de dades usant una distància (per defecte ChromaDB usa la distància euclidiana).

In [ ]:
results = collection.query(
    query_texts=["What is a DBMS?"],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)
results

Destacar també  que es poden fer consultes una mica més complexes usant la informació de les metadades (més informació [aquí](https://docs.trychroma.com/usage-guide)). Per exemple, a la següent consulta només considerem els contextos que siguin de Beyoncé.

In [ ]:
title = "Beyoncé"

results = collection.query(
    query_texts=["What is her most famous album?"],
    n_results=3,
    where={"title": title}
)
results["documents"]

En la següent consulta considerem els contextos que no siguin de Beyoncé.

In [ ]:
title = "Beyoncé"

results = collection.query(
    query_texts=["What is her most famous album?"],
    n_results=3,
    where={"title": {"$ne": title}}
)
results["documents"]

També es pot afinar la cerca fent servir el contingut dels propis documents. Per a fer-ho podem utilitzar el paràmetre `where_document` del mètode `query`.

In [ ]:
collection.query(
    query_texts=["What is her most famous album?"],
    n_results=3,
    where_document= {"$contains": "George Michael"}
)

El mètode `delete`, a més d'esborrar per identificador, també es pot fer servir per esborrar diversos documents que compleixin una condició. Per exemple, si volem esborrar tot allò relacionat amb la banda Queen.

In [ ]:
collection.delete(
    where={"title": "Queen_(band)"}
)

In [ ]:
collection.query(
    query_texts=["What is her most famous album?"],
    n_results=3,
    where={"title": "Queen_(band)"}
)

## Connectant ChromaDB amb OpenAI (generant les respostes)

Fins ara, el sistema que hem ideat (és a dir, el retriever) és força útil per si mateix. Donada una pregunta, podem recuperar contextos que potencialment tenen la resposta a la pregunta. Tanmateix, el fet de llegir els contextos i després respondre és quelcom encara un xic tediós. Així doncs, en aquest part del notebook connectarem el retriever amb GPT fent servir la llibreria [langchain](https://python.langchain.com/docs/get_started/introduction.html). GPT és un model d'OpenAI que s'encarregarà de llegir els potencials contextos i de respondre la pregunta.

Comencem instanciant el client Chroma i la función que genera els vectors (recordem que el model que fa servir per defecte ChromaDB és `all-MiniLM-L6-v2`).

**Nota.** Si ja tens la base de dades vectorial creada al directori `./database`, no és necessari que executis tot el codi anterior (a excepció del primer tros de codi: la càrrega del dataset en memòria).

In [ ]:
import chromadb

from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings

client = chromadb.PersistentClient(path="./database")
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

A continuació, instanciem un objecte de la llibreria langchain associat a la nostra bbdd de ChromaDB. Per a això, li passem com a entrada el client ChromaDB, el nom de la col·lecció i la funció que genera vectors.

In [ ]:
from langchain.vectorstores import Chroma

langchain_chroma = Chroma(
    client=client,
    collection_name="squad",
    embedding_function=embedding_function,
)
print("There are", langchain_chroma._collection.count(), "contexts in the collection")

Ara connectem ChromaDB amb OpenAI finalitzant així el nostre sistema QA. Per dur a terme aquest darrer pas, farem servir la classe de Langchain `RetrievalQA`. Abans d'instanciar l'objecte, és necessari introduir una `OPENAI_API_KEY`. Per a això us haureu de crear un compte a [OpenAI](https://openai.com/) i obtenir el vostre token.

**Nota**: Si reutilitzeu un compte d'OpenAI creat fa temps, és possible que no disposeu de crèdits, donat que aquests expiren al mes de la creació. Així doncs, si aquest es el teu cas, hauràs de crear un altre compte i generar un token amb aquest nou compte per tal de poder accedir a aquesta funcionalitat.

In [ ]:
import os

from langchain.llms import OpenAI
from langchain.chains import RetrievalQA

os.environ['OPENAI_API_KEY'] = '...'
qa = RetrievalQA.from_chain_type(llm=OpenAI(temperature=0),
                                 retriever=langchain_chroma.as_retriever())

Un cop instanciat el nostre sistema QA (l'objecte `qa`), ja podem introduir preguntes amb la funció `run`.

In [ ]:
query = "What is a DBMS?"
results = qa.run(query)
print(results)

Per veure si el nostre sistema és competent, pots escollir preguntes aleatòries del nostre dataset i veure si respon correctament.

In [ ]:
import random

i = random.randint(0, len(squad) - 1)
question = squad[i]['question']
expected_answer = squad[i]['answers']['text'][0]
results = qa.run(question)
print(f'Question: {question}')
print(f'Predicted answer: {results}')
print(f'True answer: {expected_answer}')